In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv("../data/raw/Telco_data.csv")
df = df[df["Churn Value"] == 0]
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
df.columns

In [ ]:
useless_columns = ["Count", "Country", "State", "City", "Zip Code", "Lat Long", "Latitude", "Longitude","Churn Value", "Churn Score", "Churn Label", "Churn Reason"]
df = df.drop(columns = useless_columns)

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object"])

In [ ]:
df.info()

In [ ]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Total Charges"] = df["Total Charges"].fillna(0.0)

In [ ]:
df["Total Charges"].dtype

In [ ]:
df.describe()

In [ ]:
df.describe(include = ["object"])

In [ ]:
df["Internet Service"].unique()

In [ ]:
df["is_fiber"] = (df["Internet Service"] == "Fiber optic").astype(int)

In [ ]:
df.describe()

In [ ]:
corr = df.drop(columns = ["CustomerID"]).corr(numeric_only=True)
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr, 
    annot=True,
    cmap="coolwarm", 
    fmt=".2f",
    vmin=-1, vmax=1 
)

plt.title("Correlation Matrix", fontsize=16)
plt.show()

In [ ]:
df.describe(include=["object"])

In [ ]:
object_columns  = df.select_dtypes(include='object')
for column in object_columns.columns:
    print(object_columns[column].unique())

In [ ]:
numeric_map = {
    "Yes": 1,
    "Male": 1,
    "No": 0,
    "No internet service": 0,
    "No phone service": 0,
    "Female": 0,
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2,
    "Bank transfer (automatic)": 0,
    "Credit card (automatic)": 0,
    "Electronic check": 1,
    "Mailed check": 2,
    "DSL": 1,
    "Fiber optic": 2,
}

In [ ]:
df = df.replace(numeric_map)

In [ ]:
df.info()

In [ ]:
df.describe(include=["object"])

In [ ]:
extra_services_columns = ["Phone Service", "Multiple Lines", "Online Security", "Online Backup", "Device Protection", "Streaming TV", "Streaming Movies"]

df["Total Extra Services"] = df[extra_services_columns].sum(axis = 1)


In [ ]:
service_columns = extra_services_columns.copy() + ["Total Extra Services"] + ["is_fiber"]

In [ ]:
corr_services = df[service_columns].corr()
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_services, 
    annot=True,
    cmap="coolwarm", 
    fmt=".2f",
    vmin=-1, vmax=1 
)

plt.title("Services Correlation Matrix", fontsize=16)
plt.show()


In [ ]:

corr_total = df.drop(columns = ["Internet Service", "CustomerID"]).corr()
plt.figure(figsize=(24, 20))

sns.heatmap(
    corr_total, 
    annot=True,
    cmap="coolwarm", 
    fmt=".2f",
    vmin=-1, vmax=1 
)

plt.title("Services Correlation Matrix", fontsize=16)
plt.show()

In [ ]:
corr_fiber = df.drop(columns=["Internet Service", "CustomerID"]).corr()[["is_fiber"]].sort_values(by="is_fiber", ascending=False)
plt.figure(figsize=(10, 20))

sns.heatmap(
    corr_fiber, 
    annot=True,
    cmap="coolwarm", 
    fmt=".2f",
    vmin=-1, vmax=1 
)

plt.title("Services Correlation Matrix", fontsize=16)
plt.show()

In [ ]:
df = df.drop(columns = ["Monthly Charges", "Total Charges"])

In [ ]:
X = df.drop(columns = ["is_fiber", "Internet Service", "CustomerID"])
y = df["is_fiber"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

path_processed = "../data/processed/"

X_train.to_csv(f"{path_processed}X_train.csv", index=False)
X_test.to_csv(f"{path_processed}X_test.csv", index=False)
y_train.to_csv(f"{path_processed}y_train.csv", index=False)
y_test.to_csv(f"{path_processed}y_test.csv", index=False)

In [ ]:
df_nofiber = df[df["is_fiber"] == 0].copy()
df_nofiber = df_nofiber.drop(columns = ["is_fiber", "Internet Service"])

In [ ]:
optimized_columns = [
    "CustomerID",
    'CLTV',
 'Tenure Months',
 'Total Extra Services',
 'Payment Method',
 'Multiple Lines',
 'Contract',
 'Streaming TV',
 'Phone Service',
 'Paperless Billing',
 'Streaming Movies',
 'Senior Citizen',
 'Tech Support',
 'Online Security']

df_nofiber = df_nofiber[optimized_columns]

df_nofiber.columns

In [ ]:
df_nofiber.to_csv(f"{path_processed}nf_clients.csv", index=False)